In [1]:
from utils import prepare_dataframe, merge_csv_files_in_directory

#merge_csv_files_in_directory()
df = prepare_dataframe("merged.csv")
resulting_label_count = df['labels'].nunique()
df


labels
bug            17934
enhancement    16113
clean          15336
performance     5332
refactor        3320
Name: count, dtype: int64
labels
enhancement    15300
bug            15300
clean          15300
Name: count, dtype: int64


,full_repo_name,labels,patch,index
16780,apache/skywalking,enhancement,"@@ -142,4 +142,13 @@ public class StorageModul...",16780
17669,apache/skywalking,enhancement,"@@ -21,6 +21,7 @@\n import java.util.Map;\n im...",17669
39739,apache/netbeans,enhancement,"@@ -19,9 +19,8 @@\n package org.netbeans.modul...",39739
5509,apache/myfaces-tobago,enhancement,"@@ -26,6 +26,7 @@\n \n import jakarta.faces.co...",5509
3099,apache/cloudstack,enhancement,"@@ -42,14 +42,18 @@ public final class Libvirt...",3099
...,...,...,...,...
28870,apache/bookkeeper,clean,"@@ -245,6 +245,7 @@ void unsetSuccessAndSendWr...",28870
30713,apache/netbeans,clean,"@@ -36,7 +36,6 @@\n import org.eclipse.persist...",30713
24323,apache/pulsar,clean,"@@ -18,30 +18,29 @@\n */\n package org.apache...",24323
32419,apache/netbeans,clean,"@@ -77,7 +77,7 @@ public void initialize(Templ...",32419


In [2]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "microsoft/graphcodebert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to("cuda:0")

/opt/miniconda3/envs/code1001/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
import os

embedding_directory = 'embeddings/GCB_LHS_embeddings_patch_before_largewithcodedata_15300limit3label'

os.makedirs(embedding_directory, exist_ok=True)

In [4]:
from utils import extract_code_from_patch

# Tokenizing using subtracting full code before and after embeddings

# def generate_and_save_embeddings_batch(contents_before, contents_after, indices):
def generate_and_save_embedding(patch,label, index):

    patch_before, patch_after = extract_code_from_patch(patch)
    
    
    # Tokenize the before and after code patch snippet
    tokens_before = tokenizer(patch_before, truncation=True, padding='max_length', max_length=512, return_tensors='pt').to("cuda:0")

    # Tokenize the before and after code snippets
    # tokens_before = tokenizer(list(contents_before), truncation=True, padding='max_length', max_length=512, return_tensors='pt').to("cuda:0")
    # tokens_after = tokenizer(list(contents_after), truncation=True, padding='max_length', max_length=512, return_tensors='pt').to("cuda:0")

    # Get the embeddings for both tokenized sequences
    with torch.no_grad():

        embedding_before = model(**tokens_before).last_hidden_state
        # embeddings_after = model(**tokens_after).pooler_output
    
    # Subtract the embeddings
    # subtracted_embeddings = embeddings_after - embeddings_before
    # print(subtracted_embeddings.shape)
    
    # Average the subtracted embeddings across the token dimension
    # subtracted_embeddings = subtracted_embeddings.mean(dim=1)
    
    save_path = os.path.join(embedding_directory, f"embedding_{index}_{label}.pt")
    torch.save(embedding_before, save_path)
    

In [5]:
import tqdm

for index,row in tqdm.notebook.tqdm(df.iterrows()):
    generate_and_save_embedding(row['patch'],row['labels'],row['index'])

0it [00:00, ?it/s]